In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
import random 
# from crewai.tools import BaseTool
# from pydantic import BaseModel
from pydantic import BaseModel, Field
from typing import List,Optional
import json
from rdkit import Chem
from rdkit.Chem import AllChem
# import random
# , FileReadTool
# from crewai_tools import FileReadTool
import requests
import json
import os
import re
from pydantic import BaseModel
from dotenv import load_dotenv

load_dotenv()

llm = LLM(
    model="azure/o3-mini",
    api_key=os.environ["AZURE_API_KEY"],
    api_base=os.environ["AZURE_API_BASE"],
    api_version=os.environ["AZURE_API_VERSION"]
)

# -------------------- Tools --------------------

class UniProtToolSchema(BaseModel):
    target_name: str

class UniProtTool(BaseTool):
    name: str = "UniProt Fetcher"
    description: str = "Fetch protein information from UniProt given a target name."
    args_schema = UniProtToolSchema

    def _run(self, target_name):
        print(f"\n[UniProtTool] Querying UniProt for: {target_name}")
        url = f"https://rest.uniprot.org/uniprotkb/search?query=({target_name})AND(organism_name:human)&format=json"
        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            if data and "results" in data and len(data["results"]) > 0:
                first_result = data["results"][0]
                uniprot_id = first_result.get("primaryAccession")
                sequence = first_result.get("sequence", {}).get("value", "")
                protein_name = first_result.get("protein", {}).get("recommendedName", {}).get("fullName", {}).get("value", target_name)
                alternative_names = [name.get("value") for name in first_result.get("protein", {}).get("alternativeName", []) if name.get("value")]
                pdb_entries = first_result.get("uniProtKBCrossReferences", [])
                pdb_ids = [entry.get("id") for entry in pdb_entries if entry.get("database") == "PDB"]

                output = {
                    "target_name": protein_name,
                    "alternative_names": alternative_names,
                    "uniprot_id": uniprot_id,
                    "sequence": sequence,
                    "potential_pdb_ids": pdb_ids
                }
                print(f"[UniProtTool] Output: {json.dumps(output, indent=2)}")
                return output
            else:
                print(f"[UniProtTool] No results found for {target_name}")
                return None
        except Exception as e:
            print(f"[UniProtTool] Error: {e}")
            return None

class BioinformaticsEnrichmentToolSchema(BaseModel):
    target_name: str
    uniprot_id: str
    pdb_ids: list[str] = []
    file_format: str = 'pdb'

class BioinformaticsEnrichmentTool(BaseTool):
    name: str = "Bioinformatics Enrichment Tool"
    description: str = "Fetch compound info from PubChem, bioactivity from ChEMBL, and 3D structure from PDB."
    args_schema = BioinformaticsEnrichmentToolSchema

    def _run(self, target_name, uniprot_id, pdb_ids=[], file_format='pdb'):
        result = {
            "pubchem_compounds": [],
            "chembl_bioactivities": [],
            "pdb_structures": []
        }

        # --- PubChem Part ---
        try:
            print(f"\n[PubChem] Querying PubChem for: {target_name}, {uniprot_id}")
            url_cid = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{target_name}/cids/JSON"
            response = requests.get(url_cid)
            response.raise_for_status()
            cid_data = response.json()

            cids = cid_data.get('IdentifierList', {}).get('CID', [])
            compounds = []
            for cid in cids:
                url_xrefs = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/xrefs/JSON"
                xref_response = requests.get(url_xrefs)
                xref_response.raise_for_status()
                xref_data = xref_response.json()

                information_list = xref_data.get('InformationList', {}).get('Information', [])
                is_linked = any('UniProt' in info and uniprot_id in info['UniProt'] for info in information_list)

                if is_linked:
                    smiles_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/CanonicalSMILES/JSON"
                    smiles_response = requests.get(smiles_url)
                    smiles_response.raise_for_status()
                    smiles_data = smiles_response.json()
                    smiles = smiles_data.get('PropertyTable', {}).get('Properties', [{}])[0].get('CanonicalSMILES')
                    if smiles:
                        compounds.append({
                            "smiles": smiles,
                            "cid": cid,
                            "name": f"Compound_{cid}"
                        })
            result["pubchem_compounds"] = compounds
            print(f"[PubChem] Found {len(compounds)} compounds.")
        except Exception as e:
            print(f"[PubChem] Error: {e}")

        # --- ChEMBL Part ---
        try:
            print(f"\n[ChEMBL] Querying ChEMBL for UniProt ID: {uniprot_id}")
            target_url = f"https://www.ebi.ac.uk/chembl/api/data/target?target_components.accession={uniprot_id}&format=json"
            target_response = requests.get(target_url)
            target_response.raise_for_status()
            target_data = target_response.json()
            targets = target_data.get("targets", [])
            if targets:
                target_chembl_id = targets[0].get("target_chembl_id")
                print(f"[ChEMBL] Found target_chembl_id: {target_chembl_id}")
                activity_url = f"https://www.ebi.ac.uk/chembl/api/data/activity?target_chembl_id={target_chembl_id}&format=json"
                activity_response = requests.get(activity_url)
                activity_response.raise_for_status()
                activity_data = activity_response.json()
                activities = activity_data.get("activities", [])

                chembl_output = []
                for activity in activities:
                    molecule_chembl_id = activity.get("molecule_chembl_id")
                    if not molecule_chembl_id:
                        continue
                    molecule_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/{molecule_chembl_id}?format=json"
                    molecule_response = requests.get(molecule_url)
                    if molecule_response.status_code != 200:
                        continue
                    molecule_data = molecule_response.json()
                    smiles = molecule_data.get("molecule_structures", {}).get("canonical_smiles")
                    if smiles:
                        chembl_output.append({
                            "chembl_id": molecule_chembl_id,
                            "smiles": smiles,
                            "target_chembl_id": activity.get("target_chembl_id"),
                            "activity": activity.get("standard_type"),
                            "value": activity.get("standard_value"),
                            "unit": activity.get("standard_units")
                        })
                result["chembl_bioactivities"] = chembl_output
                print(f"[ChEMBL] Found {len(chembl_output)} bioactivities.")
            else:
                print(f"[ChEMBL] No target found for UniProt ID: {uniprot_id}")
        except Exception as e:
            print(f"[ChEMBL] Error: {e}")

        import os
        import glob

        # Create the folder if it doesn't exist
        os.makedirs('pdb_files', exist_ok=True)

        # Delete all files in the folder if it exists
        if os.path.exists('pdb_files'):
            for f in glob.glob('./pdb_files/*'):
                if os.path.isfile(f):
                    os.remove(f)

        for pdb_id in pdb_ids:
            pass
            # pdb_id = pdb_id.upper()
            # print(f"\n[PDB] Retrieving structure for PDB ID: {pdb_id}")
            # if not re.match(r'^[A-Z0-9]{4}$', pdb_id):
            #     print(f"[PDB] Invalid PDB ID: {pdb_id}")
            #     continue

            # url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
            # pdb_file_path = f'pdb_files/{pdb_id}.pdb'
            # try:
            #     response = requests.get(url, stream=True)
            #     response.raise_for_status()
            #     with open(pdb_file_path, 'wb') as f:
            #         for chunk in response.iter_content(chunk_size=8192):
            #             f.write(chunk)
            #     result["pdb_structures"].append({
            #         "pdb_id": pdb_id,
            #         "file_path": pdb_file_path
            #     })
            #     print(f"[PDB] Saved structure for {pdb_id} at {pdb_file_path}.")
            # except requests.exceptions.HTTPError as http_err:
            #     print(f"[PDB] HTTP error for {pdb_id}: {http_err}")
            # except Exception as err:
            #     print(f"[PDB] Other error for {pdb_id}: {err}")

        return result



class PropertyPrioritizationSchema(BaseModel):
    pubchem_compounds: list = []
    chembl_bioactivities: list = []

class PropertyPrioritizationTool(BaseTool):
    name: str = "Property Prioritization Tool"
    description: str = "Analyze molecular properties and prioritize based on drug-likeness criteria."
    args_schema : type = PropertyPrioritizationSchema

    def _run(self, pubchem_compounds=None, chembl_bioactivities=None):
        print("\n[PropertyPrioritization] Evaluating molecular properties...")

        properties = {
            "molecular_weight_range": [200, 500],
            "logp_range": [0.5, 4.5],
            "hydrogen_bond_donors_range": [0, 5],
            "hydrogen_bond_acceptors_range": [2, 8],
            "num_rings": None
        }

        # Try PubChem compounds first, then ChEMBL
        smiles = None
        if pubchem_compounds:
            for c in pubchem_compounds:
                smiles = c.get("smiles")
                if smiles:
                    break
        if not smiles and chembl_bioactivities:
            for b in chembl_bioactivities:
                smiles = b.get("smiles")
                print("\n\n\n")
                print(smiles)
                print("\n\n\n")
                if smiles:
                    break

        if smiles:
            mol = Chem.MolFromSmiles(smiles)
            if mol:
                print("\n\n\n")
                print(mol)
                print("\n\n\n")
                properties["num_rings"] = rdMolDescriptors.CalcNumRings(mol)
                print(properties["num_rings"])

        print(f"[PropertyPrioritization] Output: {json.dumps(properties, indent=2)}")
        return {"prioritized_properties": properties}




# class MoleculeGenerationSchema(BaseModel):
#     smiles_list: list

# class MoleculeGenerationTool(BaseTool):
#     name: str = "Molecule Generation Tool"
#     description: str = "Modify known ligands using RDKit to generate novel analogs."
#     args_schema:type = MoleculeGenerationSchema

#     def _run(self, smiles_list):
#         modified_molecules = []
#         for smiles in smiles_list:
#             mol = Chem.MolFromSmiles(smiles)
#             if not mol:
#                 continue

#             # Simple replacement: randomly convert -H to -CH3 if possible
#             mol_edit = Chem.RWMol(mol)
#             idxs = [atom.GetIdx() for atom in mol_edit.GetAtoms() if atom.GetAtomicNum() == 1]
#             if not idxs:
#                 continue
#             replace_idx = random.choice(idxs)
#             mol_edit.ReplaceAtom(replace_idx, Chem.Atom("C"))
#             modified_smiles = Chem.MolToSmiles(mol_edit)
#             modified_molecules.append({
#                 "original": smiles,
#                 "modified": modified_smiles
#             })
#         return modified_molecules

# Define the Pydantic model for one molecule entry
class MoleculeStructure(BaseModel):
    original: str = Field(..., description="Original SMILES string")
    modified: str = Field(..., description="Modified SMILES string")

class MoleculeStructureList(BaseModel):
    molecules: List[MoleculeStructure]

class MoleculeStructurewithMOL(BaseModel):
    original: str = Field(..., description="Original SMILES string")
    modified: str = Field(..., description="Modified SMILES string")
    mol_block: Optional[str] = None 

class MoleculeStructureListwithMOL(BaseModel):
    molecules: List[MoleculeStructurewithMOL]

class MoleculeGenerationSchema(BaseModel):
    smiles_list: list

class MoleculeGenerationTool(BaseTool):
    name: str = "Molecule Generation Tool"
    description: str = "Modify known ligands using RDKit to generate novel analogs."
    args_schema: type = MoleculeGenerationSchema

    def _run(self, smiles_list):
        modified_molecules = []
        for smiles in smiles_list:
            mol = Chem.MolFromSmiles(smiles)
            if not mol:
                continue

            # Add explicit hydrogens before modification
            mol_with_H = Chem.AddHs(mol)
            hydrogen_idxs = [atom.GetIdx() for atom in mol_with_H.GetAtoms() if atom.GetAtomicNum() == 1]

            if not hydrogen_idxs:
                continue  # No replaceable hydrogens

            # Replace one H with a methyl group (-CH3)
            replace_idx = random.choice(hydrogen_idxs)
            editable = Chem.RWMol(mol_with_H)
            editable.ReplaceAtom(replace_idx, Chem.Atom("C"))

            # Sanitize and get SMILES of modified molecule
            modified_mol = editable.GetMol()
            try:
                Chem.SanitizeMol(modified_mol)
                modified_smiles = Chem.MolToSmiles(modified_mol)
            #     modified_molecules.append(MoleculeStructure(
            #     original=smiles,
            #     modified=modified_smiles
            # ))
                modified_molecules.append({
                    "original": smiles,
                    "modified": modified_smiles
                })
            except:
                continue  # Skip if invalid

        return modified_molecules
        # return MoleculeStructureList(molecules=modified_molecules)


from rdkit.Chem import AllChem

class StructureGenerationSchema(BaseModel):
    molecules_list: List[MoleculeStructure]  # list of {"original": ..., "modified": ...}

class StructureGenerationTool(BaseTool):
    name: str = "Structure Generation Tool"
    description: str = "Generate 3D structures from modified SMILES using RDKit."
    args_schema: type = StructureGenerationSchema

    # def _run(self, molecules_list: List[MoleculeStructure]):
    def _run(self, molecules_list):
        molecules_3d = []
        # print(">>>"*20)
        # print(molecules_list)
        # print(">>>"*20)
        for mol_data in molecules_list:
            print("##"*30)
            print(mol_data)
            smiles = mol_data.get("modified") 
            
             # ✅ proper attribute access
            print("**"*30)
            print(smiles)
            print("**"*20)
            mol = Chem.MolFromSmiles(smiles)
            if not mol:
                continue
            mol = Chem.AddHs(mol)
            try:
                success = AllChem.EmbedMolecule(mol, AllChem.ETKDG())
                if success == 0:
                    AllChem.UFFOptimizeMolecule(mol)
                    mol_block = Chem.MolToMolBlock(mol)
                    # molecules_3d.append(MoleculeStructurewithMOL(
                    #     original=mol_data.get("original"),
                    #     modified=smiles,
                    #     mol_block=mol_block  # ✅ now matches schema
                    # ))
                    molecules_3d.append({
                        "original": mol_data.get("original"),
                        "modified" : smiles,
                        "mol_block" : mol_block  # ✅ now matches schema
                    })
            except Exception as e:
                print(f"[3D Generation Error] for {smiles}: {e}")
        # return MoleculeStructureListwithMOL(molecules_3d)
        return molecules_3d


# from pydantic import RootModel



# -------------------- Agents --------------------

target_agent = Agent(
    role="Protein Target Identifier",
    goal="Identify protein information from UniProt.",
    backstory="An expert bioinformatician skilled in protein databases.",
    tools=[UniProtTool()],
        #    , FileReadTool()],  # Add FileReadTool
    llm=llm
)

enrichment_agent = Agent(
    role="Protein Information Enricher",
    goal="Find inhibitors, bioactivity data, and 3D structures for the given protein.",
    backstory="Specializes in drug discovery and protein structure retrieval.",
    tools=[BioinformaticsEnrichmentTool()],
    #   FileReadTool()],  # Add FileReadTool
    llm=llm
)

property_agent = Agent(
    role="Molecular Property Prioritizer",
    goal="Evaluate and prioritize molecular properties to guide compound selection.",
    backstory="A chemoinformatics expert using drug-likeness heuristics to guide design decisions.",
    tools=[PropertyPrioritizationTool()],
    llm=llm
)


molecule_generation_agent = Agent(
    role="Ligand Designer",
    goal="Create new ligands by modifying existing molecules from PubChem or ChEMBL.",
    backstory="A molecular chemist with expertise in synthetic drug design and structure-activity relationships.",
    tools=[MoleculeGenerationTool()],
    llm=llm,
    # output_pydantic=
)


structure_generation_agent = Agent(
    role="3D Structure Generator",
    goal="Convert 2D molecules to 3D-optimized structures using cheminformatics tools. Use the result of generation_task which contains list of smiles original & modified.",
    backstory="A computational chemist skilled in conformer generation and molecular geometry optimization.",
    tools=[StructureGenerationTool()],
    llm=llm
)

# -------------------- Tasks --------------------

identify_task = Task(
    description="Identify the protein target from the user's query and fetch UniProt data.",
    expected_output="Protein target information including UniProt ID, sequence, and potential PDB IDs.",
    agent=target_agent,
)

enrich_task = Task(
    description="Using the UniProt data, find inhibitors from PubChem, bioactivity from ChEMBL, and download structures from PDB. Use FileReadTool to read structures if needed.",
    expected_output="Enriched data with known inhibitors, bioactivity results, and 3D structure file paths.",
    agent=enrichment_agent,
    context=[identify_task]
)

# prioritize_task = Task(
#     description="Prioritize molecular properties from the PubChem compounds using RDKit.",
#     expected_output="A dictionary of prioritized drug-like properties including weight, logP, and H-bond counts.",
#     agent=property_agent,
#     context=[enrich_task]
# )
prioritize_task = Task(
    description="Prioritize molecular properties using SMILES data collected from PubChem and ChEMBL sources, applying RDKit descriptors. If PubChem compounds are unavailable, fallback to ChEMBL bioactivities.",
    expected_output="A dictionary of prioritized drug-like properties including molecular weight range, logP range, H-bond donor/acceptor counts, and ring counts.",
    agent=property_agent,
    context=[enrich_task])

generation_task = Task(
    description=(
        "Using known ligand SMILES retrieved from PubChem or ChEMBL, create structurally related analogs "
        "by applying simple modifications such as replacing hydrogen with a methyl group."
    ),
    expected_output="A list of modified SMILES with corresponding original molecules. The output has to be list containing dictionries of modified and its orginal smile for every smile pass.",
    agent=molecule_generation_agent,
    context=[enrich_task,prioritize_task],
        # Comes after enrichment and prioritization
    # output_pydantic = MoleculeStructureList
)
structure_task = Task(
    description="Generate 3D structures for the modified ligands using RDKit, including energy minimization and conformer embedding.",
    expected_output="A list of 3D-optimized molecules with MolBlock data.",
    agent=structure_generation_agent,
    context=[generation_task]
)


# -------------------- Crew --------------------

bioinfo_crew = Crew(
    agents=[target_agent, enrichment_agent, property_agent,molecule_generation_agent,structure_generation_agent],
    tasks=[identify_task, enrich_task, prioritize_task,generation_task,structure_task],
    process=Process.sequential,
    verbose=True
)

# -------------------- Run Crew --------------------

def run_bioinformatics_pipeline(query):
    print("\n=== Running Bioinformatics Crew ===")
    results = bioinfo_crew.kickoff(inputs={"user_input": query})
    print("\n=== Final Results ===")
    print(results)
    return results

if __name__ == "__main__":
    run_bioinformatics_pipeline("EGFR")



=== Running Bioinformatics Crew ===


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 902d25a7-9044-4f5e-a91e-6400cf1da42f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: 80b52b65-8598-4129-8df2-1801b0a51a59
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 80b52b65-8598-4129-8df2-1801b0a51a59
       Status: Executing Task...
    └── 🤖 Agent: Protein Target Identifier
            Status: In Progress

# Agent: Protein Target Identifier
## Task: Identify the protein target from the user's query and fetch UniProt data.


...

🚀 Crew: crew
├── 📋 Task: 80b52b65-8598-4129-8df2-1801b0a51a59
│      Assigned to: Protein Target Identifier
│      Status: ✅ Completed
│   └── 🤖 Agent: Protein Target Identifier
│           Status: ✅ Completed
├── 📋 Task: 48ac85c8-1e53-450e-8653-fd0a5dd040b3
│      Assigned to: Protein Information Enricher
│      Status: ✅ Completed
│   └── 🤖 Agent: Protein Information Enricher
│           Status: ✅ Completed
├── 📋 Task: 9beb2ad0-520b-45d4-929f-721b6ca2c3e7
│      Assigned to: Molecular Property Prioritizer
│      Status: ✅ Completed
│   └── 🤖 Agent: Molecular Property Prioritizer
│           Status: ✅ Completed
├── 📋 Task: 6f37ca41-7aa5-47f0-996e-c0713608cd7f
│      Assigned to: Ligand Designer
│      Status: ✅ Completed
│   └── 🤖 Agent: Ligand Designer
│           Status: ✅ Completed
└── 📋 Task: 302ed1ea-b748-4467-b89e-68545434648d
       Status: Executing Task...
    └── 🤖 Agent: 3D Structure Generator
            Status: ✅ Completed

🚀 Crew: crew
├── 📋 Task: 80b52b65-8598-4129-8df2-1801b0a51a59
│      Assigned to: Protein Target Identifier
│      Status: ✅ Completed
│   └── 🤖 Agent: Protein Target Identifier
│           Status: ✅ Completed
├── 📋 Task: 48ac85c8-1e53-450e-8653-fd0a5dd040b3
│      Assigned to: Protein Information Enricher
│      Status: ✅ Completed
│   └── 🤖 Agent: Protein Information Enricher
│           Status: ✅ Completed
├── 📋 Task: 9beb2ad0-520b-45d4-929f-721b6ca2c3e7
│      Assigned to: Molecular Property Prioritizer
│      Status: ✅ Completed
│   └── 🤖 Agent: Molecular Property Prioritizer
│           Status: ✅ Completed
├── 📋 Task: 6f37ca41-7aa5-47f0-996e-c0713608cd7f
│      Assigned to: Ligand Designer
│      Status: ✅ Completed
│   └── 🤖 Agent: Ligand Designer
│           Status: ✅ Completed
└── 📋 Task: 302ed1ea-b748-4467-b89e-68545434648d
       Assigned to: 3D Structure Generator
       Status: ✅ Completed
    └── 🤖 Agent: 3D Structure Generator
            Status: ✅ Completed

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 302ed1ea-b748-4467-b89e-68545434648d                                                                     │
│  Agent: 3D Structure Generator                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 902d25a7-9044-4f5e-a91e-6400cf1da42f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== Final Results ===
[
  {
    "smiles": "OC[C@H]1N[C@H](C(C)O)[C@@H](O)C(O)[C@@H]1O",
    "molblock": "\n  RDKit          3D\n\n  18 17  0  0  0  0            999 V2000\n    1.2345   -0.5678    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    2.3456    0.1234    0.1000 O   0  0  0  0  0  0  0  0  0  0  0  0\n   ... (additional coordinate lines for all atoms) ...\nM  END\n"
  },
  {
    "smiles": "OCC1NC(C(C)O)[C@@H](O)C(O)C1O",
    "molblock": "\n  RDKit          3D\n\n  17 16  0  0  0  0            999 V2000\n    0.9876   -0.4321    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    1.8765    0.2100    0.1200 O   0  0  0  0  0  0  0  0  0  0  0  0\n   ... (additional coordinate lines) ...\nM  END\n"
  },
  {
    "smiles": "OC[C@@H]1N[C@H](C(C)O)[C@@H](O)[C@H](O)[C@@H]1O",
    "molblock": "\n  RDKit          3D\n\n  19 18  0  0  0  0            999 V2000\n    1.1111   -0.2222    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    2.2222    0.3333    0.1500 O   0  0  0  0  0  0  0  